EE 344 Final Project - Data Preprocessing

In [ ]:
from google.colab import drive
import numpy as np
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import pandas as pd
import glob


In [ ]:
#------------------------------
# Load data & test train split
#------------------------------

# File paths
file1 = "/content/drive/MyDrive/Junior Year/EE 344/Final Project Data/2014.csv"
file2 = "/content/drive/MyDrive/Junior Year/EE 344/Final Project Data/2015.csv"

# Columns to keep
columns_to_keep = [
    'FL_DATE', 'OP_CARRIER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME',
    'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE', 'CANCELLED',
    'CANCELLATION_CODE', 'DIVERTED', 'DEP_DELAY'
]

# Define smaller dtypes to save memory
dtypes = {
    'OP_CARRIER': 'category',
    'ORIGIN': 'category',
    'DEST': 'category',
    'CRS_DEP_TIME': 'int32',
    'CRS_ARR_TIME': 'int32',
    'CRS_ELAPSED_TIME': 'float32',
    'DISTANCE': 'float32',
    'CANCELLED': 'int8',
    'DIVERTED': 'int8',
    'DEP_DELAY': 'float32',
    'CANCELLATION_CODE': 'category'
}

# Read CSV files with selected columns and dtypes
df1 = pd.read_csv(file1, usecols=columns_to_keep, dtype=dtypes)
df2 = pd.read_csv(file2, usecols=columns_to_keep, dtype=dtypes)

# Combine into one DataFrame
all_data = pd.concat([df1, df2], ignore_index=True)

# Free memory from intermediate DataFrames
del df1, df2

# Convert date column to datetime
all_data.loc[:, 'FL_DATE'] = pd.to_datetime(all_data['FL_DATE'])

# Remove cancelled/diverted flights to reduce size
all_data = all_data[(all_data['CANCELLED'] == 0) & (all_data['DIVERTED'] == 0)].copy()

# Sort chronologically
all_data = all_data.sort_values("FL_DATE").reset_index(drop=True)

# 70/30 chronological train/test split
split_index = int(len(all_data) * 0.7)
train_df = all_data.iloc[:split_index].copy()
test_df  = all_data.iloc[split_index:].copy()

# Check date ranges
print(train_df["FL_DATE"].min(), "to", train_df["FL_DATE"].max())
print(test_df["FL_DATE"].min(), "to", test_df["FL_DATE"].max())

2014-01-01 00:00:00 to 2015-05-29 00:00:00
2015-05-29 00:00:00 to 2015-12-31 00:00:00


In [ ]:
import numpy as np
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.preprocessing import StandardScaler

# ------------------------
# Preprocessing Pipeline
# ------------------------
# Used for all models. Depending on model needs, only certain columns will be chosen for each model's data.

def preprocess_flight_data(df):
    df = df.copy()

    # Ensure FL_DATE is datetime
    df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

    # Extract basic time features from CRS_DEP_TIME
    df['CRS_DEP_TIME'] = df['CRS_DEP_TIME'].fillna(0).astype(int)
    df['DEP_HOUR'] = df['CRS_DEP_TIME'] // 100  # 0-23 hour
    df['DEP_MIN'] = df['CRS_DEP_TIME'] % 100    # minutes if needed
    df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek  # Monday=0, Sunday=6
    df['MONTH'] = df['FL_DATE'].dt.month

    # Time-of-day label
    def get_time_of_day(hour):
        if 5 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 17:
            return 'afternoon'
        elif 17 <= hour < 21:
            return 'evening'
        else:
            return 'night'
    df['TIME_OF_DAY'] = df['DEP_HOUR'].apply(get_time_of_day)

    # Holiday flag using USFederalHolidayCalendar
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df['FL_DATE'].min(), end=df['FL_DATE' ].max())
    df['IS_HOLIDAY'] = df['FL_DATE'].isin(holidays).astype(int)

    # Cyclic encoding placeholders (for linear models)
    df['DEP_HOUR_SIN'] = np.sin(2 * np.pi * df['DEP_HOUR']/24)
    df['DEP_HOUR_COS'] = np.cos(2 * np.pi * df['DEP_HOUR']/24)
    df['DAY_OF_WEEK_SIN'] = np.sin(2 * np.pi * df['DAY_OF_WEEK']/7)
    df['DAY_OF_WEEK_COS'] = np.cos(2 * np.pi * df['DAY_OF_WEEK']/7)

    return df

# Apply preprocessing to all data.
train_data = preprocess_flight_data(train_df)
test_data = preprocess_flight_data(test_df)

print(train_data.head())

     FL_DATE OP_CARRIER ORIGIN DEST  CRS_DEP_TIME  DEP_DELAY  CRS_ARR_TIME  \
0 2014-01-01         AA    ICT  DFW          1135        9.0          1300   
1 2014-01-01         UA    SAN  SFO          1122       18.0          1257   
2 2014-01-01         UA    SFO  LAX          1631       -7.0          1804   
3 2014-01-01         UA    IAH  LAX          1318       25.0          1514   
4 2014-01-01         UA    AUS  DEN           751       -4.0           906   

   CANCELLED CANCELLATION_CODE  DIVERTED  ...  DEP_HOUR  DEP_MIN  DAY_OF_WEEK  \
0          0               NaN         0  ...        11       35            2   
1          0               NaN         0  ...        11       22            2   
2          0               NaN         0  ...        16       31            2   
3          0               NaN         0  ...        13       18            2   
4          0               NaN         0  ...         7       51            2   

   MONTH  TIME_OF_DAY  IS_HOLIDAY DEP_HOUR_S

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# ------------------------
# Memory-efficient prepare_model_data
# ------------------------

def prepare_model_data(df, approach='binary', model_type='linear'):
    """
    Prepares X and y for flight delay prediction.

    Linear models: uses sparse one-hot encoding + scaled numeric + cyclic features
    Tree models: uses categorical codes + numeric features
    """
    df = df.copy()  # Only one copy at start

    # -------------------------
    # Target column
    # -------------------------
    if approach == 'binary':
        y = (df['DEP_DELAY'] > 15).astype(int)
    elif approach == 'regression':
        y = df['DEP_DELAY'].astype(float)
    elif approach == 'multiclass':
        bins = [0, 15, 30, 60, np.inf]
        labels = [0, 1, 2, 3]
        y = pd.cut(df['DEP_DELAY'], bins=bins, labels=labels, right=True).astype(int)
    else:
        raise ValueError("Approach must be 'binary', 'regression', or 'multiclass'")

    # -------------------------
    # Linear models
    # -------------------------
    if model_type == 'linear':
        # Column selection
        categorical_cols_linear = ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK']
        numeric_cols_linear = ['DISTANCE', 'CRS_ELAPSED_TIME'] + ['DEP_HOUR_SIN', 'DEP_HOUR_COS', 'DAY_OF_WEEK_SIN', 'DAY_OF_WEEK_COS']

        # Sparse one-hot encoder for categorical features
        ohe = OneHotEncoder(drop='first', sparse_output=True, handle_unknown='ignore')
        X_cat_sparse = ohe.fit_transform(df[categorical_cols_linear])

        # Scale numeric features
        scaler = StandardScaler()
        X_num_scaled = scaler.fit_transform(df[numeric_cols_linear])

        # Combine numeric + categorical (sparse)
        from scipy.sparse import hstack
        X = hstack([X_num_scaled, X_cat_sparse], format='csr')

    # -------------------------
    # Tree models
    # -------------------------
    elif model_type == 'tree':
        tree_categorical_cols = ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK', 'TIME_OF_DAY']
        tree_numeric_cols = ['DISTANCE', 'CRS_ELAPSED_TIME']

        # Convert categorical columns to category codes (memory-efficient)
        X_cat = df[tree_categorical_cols].apply(lambda x: x.astype('category').cat.codes)
        X_num = df[tree_numeric_cols]

        # Concatenate numeric + categorical codes
        X = pd.concat([X_num, X_cat], axis=1)

    else:
        raise ValueError("model_type must be 'linear' or 'tree'")

    return X, y

In [ ]:
# Binary -- For example. Each approach will be put in it's own notebook due to RAM limitations, but they are shown below commented out.
# Linear model (sparse one-hot)
X_train_linear_bin, y_train_bin = prepare_model_data(train_data, approach='binary', model_type='linear')
X_test_linear_bin, y_test_bin = prepare_model_data(test_data, approach='binary', model_type='linear')

# Tree model (categorical codes)
X_train_tree_bin, y_train_bin_tree = prepare_model_data(train_data, approach='binary', model_type='tree')
X_test_tree_bin, y_test_bin_tree = prepare_model_data(test_data, approach='binary', model_type='tree')

# Regression
# Linear model
#X_train_linear_reg, y_train_reg = prepare_model_data(train_data, approach='regression', model_type='linear')
#X_test_linear_reg, y_test_reg = prepare_model_data(test_data, approach='regression', model_type='linear')

# Tree model
#X_train_tree_reg, y_train_reg_tree = prepare_model_data(train_data, approach='regression', model_type='tree')
#X_test_tree_reg, y_test_reg_tree = prepare_model_data(test_data, approach='regression', model_type='tree')

# Multiclass
# Linear model
#X_train_linear_multi, y_train_multi = prepare_model_data(train_data, approach='multiclass', model_type='linear')
#X_test_linear_multi, y_test_multi = prepare_model_data(test_data, approach='multiclass', model_type='linear')

# Tree model
#X_train_tree_multi, y_train_multi_tree = prepare_model_data(train_data, approach='multiclass', model_type='tree')
#X_test_tree_multi, y_test_multi_tree = prepare_model_data(test_data, approach='multiclass', model_type='tree')